In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import catboost
import lightgbm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mse
from sklearn.preprocessing import LabelEncoder

In [2]:
train_path = "/kaggle/input/playground-series-s3e2/train.csv"
test_path = "/kaggle/input/playground-series-s3e2/test.csv"
sub_path = "/kaggle/input/playground-series-s3e2/sample_submission.csv"
ext_path = "/kaggle/input/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv"

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
ext_df = pd.read_csv(ext_path)
sub_df = pd.read_csv(sub_path)

In [4]:
train_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,0,Male,28.0,0,0,Yes,Private,Urban,79.53,31.1,never smoked,0
1,1,Male,33.0,0,0,Yes,Private,Rural,78.44,23.9,formerly smoked,0
2,2,Female,42.0,0,0,Yes,Private,Rural,103.00,40.3,Unknown,0
3,3,Male,56.0,0,0,Yes,Private,Urban,64.87,28.8,never smoked,0
4,4,Female,24.0,0,0,No,Private,Rural,73.36,28.8,never smoked,0


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15304 entries, 0 to 15303
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 15304 non-null  int64  
 1   gender             15304 non-null  object 
 2   age                15304 non-null  float64
 3   hypertension       15304 non-null  int64  
 4   heart_disease      15304 non-null  int64  
 5   ever_married       15304 non-null  object 
 6   work_type          15304 non-null  object 
 7   Residence_type     15304 non-null  object 
 8   avg_glucose_level  15304 non-null  float64
 9   bmi                15304 non-null  float64
 10  smoking_status     15304 non-null  object 
 11  stroke             15304 non-null  int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 1.4+ MB


In [6]:
train_df.work_type.value_counts()

Private          9752
children         2038
Self-employed    1939
Govt_job         1533
Never_worked       42
Name: work_type, dtype: int64

In [7]:
train_df.Residence_type.value_counts()

Rural    7664
Urban    7640
Name: Residence_type, dtype: int64

In [8]:
train_df.smoking_status.value_counts()

never smoked       6281
Unknown            4543
formerly smoked    2337
smokes             2143
Name: smoking_status, dtype: int64

In [9]:
cols = train_df.columns

In [10]:
ext_df = ext_df[cols]

In [11]:
train_df = pd.concat([train_df, ext_df],axis=0).drop_duplicates()

In [12]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20414 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 20414 non-null  int64  
 1   gender             20414 non-null  object 
 2   age                20414 non-null  float64
 3   hypertension       20414 non-null  int64  
 4   heart_disease      20414 non-null  int64  
 5   ever_married       20414 non-null  object 
 6   work_type          20414 non-null  object 
 7   Residence_type     20414 non-null  object 
 8   avg_glucose_level  20414 non-null  float64
 9   bmi                20213 non-null  float64
 10  smoking_status     20414 non-null  object 
 11  stroke             20414 non-null  int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 2.0+ MB


In [13]:
cols

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

In [14]:
features = ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married','work_type', 'Residence_type', 'avg_glucose_level', 'bmi','smoking_status']
target = 'stroke'

In [15]:
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for feature in cat_features:
    le = LabelEncoder()
    train_df[feature] = le.fit_transform(train_df[feature])
    test_df[feature] = le.transform(test_df[feature])

In [16]:
from optuna.integration import LightGBMPruningCallback
import optuna
from sklearn.metrics import log_loss, auc, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [17]:
def objective(trial):
    k =10
    kfold = KFold(k,shuffle=True, random_state=42)
    val_scores = []
    test_preds= []
    cv_scores = np.empty(10)

    param_grid = {
            "n_estimators": trial.suggest_int("n_estimators",  50, 10000, step = 20),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 3000, step=20),
            "max_depth": trial.suggest_int("max_depth", 3, 18),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 10000, step=20),
            "lambda_l1": trial.suggest_int("lambda_l1", 0, 100, step=5),
            "lambda_l2": trial.suggest_int("lambda_l2", 0, 100, step=5),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0, 15),
            "bagging_fraction": trial.suggest_float(
                "bagging_fraction", 0.2, 0.95, step=0.1
            ),
            "bagging_freq": trial.suggest_categorical("bagging_freq", [1]),
            "feature_fraction": trial.suggest_float(
                "feature_fraction", 0.2, 0.95, step=0.1
            ),
            'num_iterations': trial.suggest_int("num_iterations", 100, 10000, step = 20),
            'verbosity': -1
        }


    for i,(train_idxs,val_idxs) in enumerate(kfold.split(train_df)):

        X_train = train_df.iloc[train_idxs][features]
        y_train = train_df.iloc[train_idxs][target]
        X_val = train_df.iloc[val_idxs][features]
        y_val = train_df.iloc[val_idxs][target]
        params= {
         'learning_rate':0.02,
         'lambda_l1': 1.945,
         'num_leaves': 87,
         'feature_fraction': 0.79,
         'bagging_fraction': 0.93,
         'bagging_freq': 4,
         'min_data_in_leaf': 103,
         'max_depth': 17,
         'num_iterations':10000
        }

        model = lightgbm.LGBMRegressor(**param_grid)    

        model.fit(X= X_train,
                  y= y_train,
                  eval_set = (X_val,y_val),
                  early_stopping_rounds = 100,
                  eval_metric="binary_logloss",
                  verbose=False,
                  callbacks=[
                    LightGBMPruningCallback(trial, "binary_logloss")
                  ], 
                 )
        preds = model.predict(X_val)
        rmse = mse(y_val, preds,squared=False)
        val_scores.append(rmse)
        cv_scores[i] = log_loss(y_val, preds)
    #     print(f'=== Fold {i} RMSE {rmse} ====')

        preds = model.predict(test_df[features])
        test_preds.append(preds)
        

    return np.mean(cv_scores)

In [18]:
study = optuna.create_study(direction="minimize", study_name="LGBM Classifier")
func = lambda trial: objective(trial)
study.optimize(func, n_trials=20)

[I 2023-01-10 10:55:29,505] A new study created in memory with name: LGBM Classifier


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_gain_to_split is set=13.096582187112315, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=13.096582187112315
[LightGBM] [Warning] min_data_in_leaf is set=2800, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2800
[LightGBM] [Warning] lambda_l2 is set=70, reg_lambda=0.0 will be ignored. Current value: lambda_l2=70
[LightGBM] [Warning] bagging_fraction is set=0.2, subsample=1.0 will be ignored. Current value: bagging_fraction=0.2
[LightGBM] [Warning] lambda_l1 is set=75, reg_alpha=0.0 will be ignored. Current value: lambda_l1=75
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, 

[I 2023-01-10 10:55:30,262] Trial 0 finished with value: 0.17790364324434776 and parameters: {'n_estimators': 1530, 'learning_rate': 0.18409156984823197, 'num_leaves': 20, 'max_depth': 12, 'min_data_in_leaf': 2800, 'lambda_l1': 75, 'lambda_l2': 70, 'min_gain_to_split': 13.096582187112315, 'bagging_fraction': 0.2, 'bagging_freq': 1, 'feature_fraction': 0.8, 'num_iterations': 9300}. Best is trial 0 with value: 0.17790364324434776.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_gain_to_split is set=13.096582187112315, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=13.096582187112315
[LightGBM] [Warning] min_data_in_leaf is set=2800, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2800
[LightGBM] [Warning] lambda_l2 is set=70, reg_lambda=0.0 will be ignored. Current value: lambda_l2=70
[LightGBM] [Warning] bagging_fraction is set=0.2, subsample=1.0 will be ignored. Current value: bagging_fraction=0.2
[LightGBM] [Warning] lambda_l1 is set=75, reg_alpha=0.0 will be ignored. Current value: lambda_l1=75
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=1, 

[I 2023-01-10 10:55:31,729] Trial 1 finished with value: 0.13764051702816932 and parameters: {'n_estimators': 4570, 'learning_rate': 0.1806494023668779, 'num_leaves': 2640, 'max_depth': 16, 'min_data_in_leaf': 1180, 'lambda_l1': 10, 'lambda_l2': 80, 'min_gain_to_split': 0.827674778538654, 'bagging_fraction': 0.9, 'bagging_freq': 1, 'feature_fraction': 0.7, 'num_iterations': 8680}. Best is trial 1 with value: 0.13764051702816932.


...

[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_gain_to_split is set=3.994705299807517, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=3.994705299807517
[LightGBM] [Warning] min_data_in_leaf is set=6940, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6940
[LightGBM] [Warning] lambda_l2 is set=15, reg_lambda=0.0 will be ignored. Current value: lambda_l2=15
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] lambda_l1 is set=30, reg_alpha=0.0 will be ignored. Current value: lambda_l1=30
[LightGBM] [Warning] feature_fraction is set=0.4, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, su

[I 2023-01-10 10:55:44,476] Trial 18 pruned. Trial was pruned at iteration 146.


[LightGBM] [Warning] feature_fraction is set=0.4, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.4
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_gain_to_split is set=8.360783979526426, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=8.360783979526426
[LightGBM] [Warning] min_data_in_leaf is set=1200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1200
[LightGBM] [Warning] lambda_l2 is set=60, reg_lambda=0.0 will be ignored. Current value: lambda_l2=60
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=65, reg_alpha=0.0 will be ignored. Current value: lambda_l1=65
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=1, su

[I 2023-01-10 10:55:45,670] Trial 19 finished with value: 0.17790364324434776 and parameters: {'n_estimators': 3970, 'learning_rate': 0.2327057208997545, 'num_leaves': 120, 'max_depth': 14, 'min_data_in_leaf': 5360, 'lambda_l1': 0, 'lambda_l2': 45, 'min_gain_to_split': 11.235192687192091, 'bagging_fraction': 0.4, 'bagging_freq': 1, 'feature_fraction': 0.7, 'num_iterations': 3380}. Best is trial 1 with value: 0.13764051702816932.


[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_gain_to_split is set=11.235192687192091, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=11.235192687192091
[LightGBM] [Warning] min_data_in_leaf is set=5360, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5360
[LightGBM] [Warning] lambda_l2 is set=45, reg_lambda=0.0 will be ignored. Current value: lambda_l2=45
[LightGBM] [Warning] bagging_fraction is set=0.4, subsample=1.0 will be ignored. Current value: bagging_fraction=0.4
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0


In [19]:
print(f"\tBest value (rmse): {study.best_value:.5f}")
print(f"\tBest params:")

for key, value in study.best_params.items():
    print(f"\t\t{key}: {value}")

	Best value (rmse): 0.13764
	Best params:
		n_estimators: 4570
		learning_rate: 0.1806494023668779
		num_leaves: 2640
		max_depth: 16
		min_data_in_leaf: 1180
		lambda_l1: 10
		lambda_l2: 80
		min_gain_to_split: 0.827674778538654
		bagging_fraction: 0.9
		bagging_freq: 1
		feature_fraction: 0.7
		num_iterations: 8680


In [20]:
params = {
'learning_rate':0.02,
'lambda_l1': 1.945,
'num_leaves': 87,
'feature_fraction': 0.79,
'bagging_fraction': 0.93,
'bagging_freq': 4,
'min_data_in_leaf': 103,
'max_depth': 17,
'num_iterations':10000
}

In [21]:
model = lightgbm.LGBMRegressor(**study.best_params)    

model.fit(X= train_df[features],
          y= train_df[target],
          verbose=False
         )

predictions = model.predict(test_df[features])

[LightGBM] [Warning] lambda_l1 is set=10, reg_alpha=0.0 will be ignored. Current value: lambda_l1=10
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_gain_to_split is set=0.827674778538654, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.827674778538654
[LightGBM] [Warning] lambda_l2 is set=80, reg_lambda=0.0 will be ignored. Current value: lambda_l2=80
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] min_data_in_leaf is set=1180, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1180
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


In [22]:
# predictions = np.mean(np.vstack(test_preds), axis=0)
sub_df['stroke'] = predictions
sub_df.to_csv('submission.csv', index=False)

In [23]:
sub_df.head()

,id,stroke
0,15304,0.053620
1,15305,0.166320
2,15306,-0.000777
3,15307,0.038219
4,15308,0.009241


In [24]:
model.feature_importances_

array([ 0, 23,  0,  0,  1,  1,  0,  6,  1,  0], dtype=int32)

In [25]:
model.feature_name_

['gender',
 'age',
 'hypertension',
 'heart_disease',
 'ever_married',
 'work_type',
 'Residence_type',
 'avg_glucose_level',
 'bmi',
 'smoking_status']

In [26]:
for k, v in zip (model.feature_name_, model.feature_importances_):
    print(k, ":", v)

gender : 0
age : 23
hypertension : 0
heart_disease : 0
ever_married : 1
work_type : 1
Residence_type : 0
avg_glucose_level : 6
bmi : 1
smoking_status : 0
